# From a grid picture to an optimized design, with qBraid AI

1. **qBraid AI Gateway** (a vision model, called with your qBraid API key) reads a picture of a utility network and returns its sites and links as JSON.
2. The **exact optimizer** from `3g` chooses where QKD goes, and adds backup routes.
3. The result is drawn, and saved for the QKD Grid Optimizer page (`qkd_grid_optimizer.html`, Import tab) and for `3h` (run the design on qBraid).

Use your own image, or the included `network_design/sample_grid.png`.

> **Kernel:** qBraid-SDK (v0.12.2). Gateway calls use your qBraid AI allowance.

In [ ]:
import os, json, base64, getpass, subprocess, sys, requests
GW = "https://api-v2.qbraid.com/api/v1/ai"
KEY = os.environ.get("QBRAID_API_KEY") or getpass.getpass("qBraid API key (account.qbraid.com > API keys): ")
H = {"X-API-Key": KEY}
models = requests.get(f"{GW}/models", headers=H, timeout=60).json().get("data", [])
def caps(m): return json.dumps(m.get("_qbraid", {}).get("capabilities", m.get("capabilities", ""))).lower()
vision = [m["id"] for m in models if "image" in caps(m) or "vision" in caps(m)]
print(len(models), "models;", len(vision), "accept images:", vision[:10])
MODEL = next((m for m in vision if "claude" in m.lower()), vision[0] if vision else None)
print("Using:", MODEL)

In [ ]:
IMAGE = "network_design/sample_grid.png"          # change to your own grid picture
from PIL import Image
W, Hpx = Image.open(IMAGE).size
PROMPT = f'''Read this utility network diagram (image is {W} x {Hpx} pixels). List every site and every line between sites as JSON:
{{"nodes":[{{"id":"<name>","mw":<load MW or value, 20 if unknown>,"kv":<kV, 115 if unknown>,"x":<pixel x>,"y":<pixel y>}}],
 "edges":[{{"a":"<id>","b":"<id>","km":<length in km if shown, else estimate from the scale bar>}}],
 "cc":["<main control center id>","<backup control center id>"]}}
Output only the JSON.'''
b64 = base64.b64encode(open(IMAGE, "rb").read()).decode()
mt = "image/png" if IMAGE.lower().endswith("png") else "image/jpeg"
r = requests.post(f"{GW}/v1/messages", headers={**H, "anthropic-version": "2023-06-01", "content-type": "application/json"}, timeout=300,
                  json={"model": MODEL, "max_tokens": 8000, "messages": [{"role": "user", "content": [
                      {"type": "image", "source": {"type": "base64", "media_type": mt, "data": b64}}, {"type": "text", "text": PROMPT}]}]})
if r.status_code != 200:     # model may only speak the OpenAI format
    r = requests.post(f"{GW}/chat/completions", headers={**H, "content-type": "application/json"}, timeout=300,
                      json={"model": MODEL, "messages": [{"role": "user", "content": [
                          {"type": "image_url", "image_url": {"url": f"data:{mt};base64,{b64}"}}, {"type": "text", "text": PROMPT}]}]})
    text = r.json()["choices"][0]["message"]["content"]
else:
    text = "".join(c.get("text", "") for c in r.json()["content"])
grid = json.loads(text[text.index("{"): text.rindex("}") + 1])
print(f"qBraid AI read {len(grid['nodes'])} sites and {len(grid['edges'])} links; control centers {grid.get('cc')}")
json.dump(grid, open("grid_from_image.json", "w"), indent=1)

## Optimize the network the AI read

The edges are converted to the optimizer's format, with the control centers the AI marked as the candidates. The exact integer program (as in `3g`) then picks 8 QKD links and up to 4 trusted nodes.

In [ ]:
region = {"subs": [n["id"] for n in grid["nodes"]], "core": grid.get("cc") or [n["id"] for n in grid["nodes"]],
          "links": [[[e["a"], e["b"]], float(e["km"]), 115.0] for e in grid["edges"]],
          "load": {n["id"]: float(n.get("mw", 20)) for n in grid["nodes"]}, "kv": {n["id"]: float(n.get("kv", 115)) for n in grid["nodes"]}}
os.makedirs("network_design/regions", exist_ok=True)
json.dump(region, open("network_design/regions/from_image.json", "w"))
out = subprocess.run([sys.executable, "network_design/final_design.py", "network_design/regions/from_image.json",
                      "image_design.png", "image_design.json"], capture_output=True, text=True)
print(out.stdout[-2500:] or out.stderr[-2500:])
from IPython.display import Image as Img, display
display(Img("image_design.png", width=900))

**Next:** load `grid_from_image.json` in the planner's Import tab to explore it with the sliders, or press the planner's *Download design JSON* and run `3h` to test every QKD link on qBraid's GPU simulator. Check the AI's reading against the picture: vision models can miss or invent a line, so the planner's edit tools are there to fix it.